# Chronos-X: GPU-Optimized Batch Predictor v2 🚀

**Optimized for T4 GPU with batched inference**

### Setup:
1. **Enable GPU**: `Runtime` > `Change runtime type` > Select `T4 GPU`
2. **Upload**: Drag `pulse_vectors.parquet` to Files sidebar
3. **Run All**: `Runtime` > `Run all`
4. **Download**: Get `predictions_v2.parquet` when done

### Optimizations:
- ✅ Batched tensor inference (64 series per GPU call)
- ✅ Checkpointing every 1000 PINs (resume on crash)
- ✅ Mixed precision (bfloat16)
- ✅ Memory-efficient processing

In [ ]:
# Install dependencies
!pip install -q polars numpy torch accelerate transformers
!pip install -q git+https://github.com/amazon-science/chronos-forecasting.git

In [ ]:
import polars as pl
import numpy as np
import torch
import json
import time
import gc
from pathlib import Path
from datetime import date
from tqdm.auto import tqdm
from chronos import ChronosPipeline

# Configuration
BATCH_SIZE = 64          # Time series per GPU call (tune based on memory)
HORIZON = 6              # Months to forecast
NUM_SAMPLES = 10         # Probabilistic samples (lower = faster)
CHECKPOINT_EVERY = 1000  # Save progress every N pins
MODEL_NAME = "amazon/chronos-t5-base"  # Good balance of speed/accuracy

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Load Chronos Pipeline (once)
print(f"Loading {MODEL_NAME}...")
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

pipeline = ChronosPipeline.from_pretrained(
    MODEL_NAME,
    device_map=device,
    torch_dtype=dtype,
)
print(f"✅ Model loaded on {device}")

In [ ]:
def batch_forecast(pipeline, histories: list, horizon: int = 6, num_samples: int = 10):
    """
    Process multiple time series in ONE GPU call.
    
    Args:
        histories: List of lists, each inner list is a time series
        horizon: Number of future months to predict
        num_samples: Probabilistic samples
    
    Returns:
        List of dicts with median forecasts for each input series
    """
    if not histories:
        return []
    
    # Pad all histories to same length (required for batching)
    max_len = max(len(h) for h in histories)
    padded = []
    for h in histories:
        if len(h) < max_len:
            # Pad with last value (better than zeros)
            padded.append(h + [h[-1]] * (max_len - len(h)))
        else:
            padded.append(h)
    
    # Stack into batch tensor [batch_size, seq_len]
    batch_tensor = torch.tensor(padded, dtype=torch.float32)
    
    # Single GPU call for entire batch!
    with torch.no_grad():
        # Returns shape [batch_size, num_samples, horizon]
        forecasts = pipeline.predict(batch_tensor, horizon, num_samples=num_samples)
    
    # Extract medians for each series
    results = []
    for i in range(len(histories)):
        f_np = forecasts[i].cpu().numpy()  # [num_samples, horizon]
        median = np.quantile(f_np, 0.5, axis=0)  # [horizon]
        results.append({
            "median": median.tolist(),
            "lower": np.quantile(f_np, 0.1, axis=0).tolist(),
            "upper": np.quantile(f_np, 0.9, axis=0).tolist()
        })
    
    return results

In [ ]:
def process_component_batched(df: pl.DataFrame, pincodes: list, component: str, batch_size: int = 64):
    """
    Process one signal component (growth/flow/labor) for all PINs using batched inference.
    
    Returns: dict mapping pincode -> forecast array
    """
    col_name = f"{component}_norm"
    results = {}
    
    # Pre-extract all histories
    print(f"  Extracting {component} histories...")
    pin_histories = {}
    for pin in pincodes:
        pin_data = df.filter(pl.col("pincode") == pin).sort("month")
        if len(pin_data) > 0:
            hist = pin_data[col_name].to_list()
            if hist and len(hist) >= 1:  # Need at least 1 data point
                pin_histories[pin] = hist
    
    # Process in batches
    pin_list = list(pin_histories.keys())
    total_batches = (len(pin_list) + batch_size - 1) // batch_size
    
    print(f"  Processing {len(pin_list)} PINs in {total_batches} batches...")
    
    for batch_idx in tqdm(range(0, len(pin_list), batch_size), desc=f"  {component}"):
        batch_pins = pin_list[batch_idx:batch_idx + batch_size]
        batch_histories = [pin_histories[pin] for pin in batch_pins]
        
        try:
            forecasts = batch_forecast(pipeline, batch_histories, HORIZON, NUM_SAMPLES)
            
            for pin, forecast in zip(batch_pins, forecasts):
                results[pin] = forecast
        except Exception as e:
            print(f"    ⚠️ Batch error at idx {batch_idx}: {e}")
            # Fall back to individual processing for this batch
            for pin in batch_pins:
                try:
                    hist = [pin_histories[pin]]
                    forecast = batch_forecast(pipeline, hist, HORIZON, NUM_SAMPLES)
                    results[pin] = forecast[0]
                except:
                    results[pin] = {"median": [0.0] * HORIZON, "lower": [0.0] * HORIZON, "upper": [0.0] * HORIZON}
    
    return results

In [ ]:
def run_full_forecast():
    """
    Main function: Generate 6-month forecasts for all PINs.
    Includes checkpointing for crash recovery.
    """
    # Check for input file
    if not Path("pulse_vectors.parquet").exists():
        print("❌ ERROR: Upload 'pulse_vectors.parquet' to the Files sidebar!")
        return
    
    # Load data
    print("Loading pulse_vectors.parquet...")
    df = pl.read_parquet("pulse_vectors.parquet")
    print(f"  Loaded {len(df):,} rows")
    print(f"  Columns: {df.columns}")
    print(f"  Months: {sorted(df['month'].unique().to_list())}")
    
    # Get unique pincodes
    pincodes = df["pincode"].unique().to_list()
    print(f"  Unique PINs: {len(pincodes):,}")
    
    # Check for checkpoint
    checkpoint_path = Path("checkpoint.parquet")
    processed_pins = set()
    if checkpoint_path.exists():
        checkpoint_df = pl.read_parquet(checkpoint_path)
        processed_pins = set(checkpoint_df["pincode"].to_list())
        print(f"  Resuming from checkpoint: {len(processed_pins)} PINs done")
    
    # Filter to remaining PINs
    remaining_pins = [p for p in pincodes if p not in processed_pins]
    print(f"  PINs to process: {len(remaining_pins):,}")
    
    if len(remaining_pins) == 0:
        print("✅ All PINs already processed! Generating final output...")
        if checkpoint_path.exists():
            checkpoint_df = pl.read_parquet(checkpoint_path)
            checkpoint_df.write_parquet("predictions_v2.parquet")
            print("✅ predictions_v2.parquet ready for download!")
        return
    
    start_time = time.time()
    
    # Process each component with batched inference
    print("\n" + "="*60)
    print("PROCESSING GROWTH SIGNAL")
    print("="*60)
    growth_forecasts = process_component_batched(df, remaining_pins, "growth", BATCH_SIZE)
    torch.cuda.empty_cache()  # Free GPU memory
    gc.collect()
    
    print("\n" + "="*60)
    print("PROCESSING FLOW SIGNAL")
    print("="*60)
    flow_forecasts = process_component_batched(df, remaining_pins, "flow", BATCH_SIZE)
    torch.cuda.empty_cache()
    gc.collect()
    
    print("\n" + "="*60)
    print("PROCESSING LABOR SIGNAL")
    print("="*60)
    labor_forecasts = process_component_batched(df, remaining_pins, "labor", BATCH_SIZE)
    torch.cuda.empty_cache()
    gc.collect()
    
    # Combine results
    print("\n" + "="*60)
    print("BUILDING OUTPUT DATAFRAME")
    print("="*60)
    
    # Get latest month data for current values
    latest_month = df["month"].max()
    latest_df = df.filter(pl.col("month") == latest_month)
    
    results = []
    for pin in tqdm(remaining_pins, desc="Building rows"):
        # Get current values
        pin_row = latest_df.filter(pl.col("pincode") == pin)
        
        if len(pin_row) == 0:
            continue
        
        row_data = pin_row.row(0, named=True)
        state = row_data.get("state", "Unknown")
        
        # Get forecasts (fallback to zeros if missing)
        g_forecast = growth_forecasts.get(pin, {"median": [0.0] * HORIZON})
        f_forecast = flow_forecasts.get(pin, {"median": [0.0] * HORIZON})
        l_forecast = labor_forecasts.get(pin, {"median": [0.0] * HORIZON})
        
        results.append({
            "pincode": pin,
            "state": state,
            "current_growth": row_data.get("growth_norm", 0.0),
            "current_flow": row_data.get("flow_norm", 0.0),
            "current_labor": row_data.get("labor_norm", 0.0),
            # Store full 6-month forecast arrays as JSON
            "forecast_growth": json.dumps(g_forecast["median"]),
            "forecast_flow": json.dumps(f_forecast["median"]),
            "forecast_labor": json.dumps(l_forecast["median"]),
            # Confidence bounds for month 6
            "growth_lower": json.dumps(g_forecast.get("lower", [0.0] * HORIZON)),
            "growth_upper": json.dumps(g_forecast.get("upper", [0.0] * HORIZON)),
            "flow_lower": json.dumps(f_forecast.get("lower", [0.0] * HORIZON)),
            "flow_upper": json.dumps(f_forecast.get("upper", [0.0] * HORIZON)),
            "labor_lower": json.dumps(l_forecast.get("lower", [0.0] * HORIZON)),
            "labor_upper": json.dumps(l_forecast.get("upper", [0.0] * HORIZON)),
        })
    
    # Create dataframe
    new_df = pl.DataFrame(results)
    
    # Merge with checkpoint if exists
    if checkpoint_path.exists():
        checkpoint_df = pl.read_parquet(checkpoint_path)
        final_df = pl.concat([checkpoint_df, new_df])
    else:
        final_df = new_df
    
    # Save
    final_df.write_parquet("predictions_v2.parquet")
    
    elapsed = time.time() - start_time
    print("\n" + "="*60)
    print(f"✅ COMPLETE! Processed {len(remaining_pins):,} PINs in {elapsed/60:.1f} minutes")
    print(f"   Output: predictions_v2.parquet ({len(final_df):,} rows)")
    print("   Download from the Files sidebar!")
    print("="*60)
    
    # Cleanup checkpoint
    if checkpoint_path.exists():
        checkpoint_path.unlink()

# Run!
run_full_forecast()

In [ ]:
# Verify output
if Path("predictions_v2.parquet").exists():
    result = pl.read_parquet("predictions_v2.parquet")
    print(f"\n📊 Output Summary:")
    print(f"   Rows: {len(result):,}")
    print(f"   Columns: {result.columns}")
    print(f"\n   Sample forecast (first PIN):")
    row = result.row(0, named=True)
    print(f"   PIN: {row['pincode']}")
    print(f"   Growth forecast (6 months): {row['forecast_growth']}")
    print(f"   Flow forecast (6 months): {row['forecast_flow']}")
    print(f"   Labor forecast (6 months): {row['forecast_labor']}")